<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SFT

## Loading libraries

In [1]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 98 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (85.7 kB/s)0m

78Selecting previously unselected package tree.
(Reading database ... 129073 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...............................................] 8Unpacking tree (2.0.2-1) ...
7Progress: [ 40%] [#######################...................................] 8Setting up tree (2.0.2-1) ...
7Progress: [ 60%] [##################################........................] 

In [2]:
!pip install peft transformers[torch] trl bitsandbytes datasets wandb -U --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 103.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.3 MB/s eta 0:00:0000:010:01


In [3]:
import os
import random, math
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

2025-12-25 08:50:23.703386: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766652623.883753      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766652623.935465      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766652624.360166      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766652624.360204      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766652624.360206      55 computation_placer.cc:177] computation placer alr

In [4]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [6]:
shuffled_dataset = dataset.shuffle(seed=42)
split_ds_1 = shuffled_dataset.train_test_split(test_size=0.1)
split_ds_2 = split_ds_1['test'].train_test_split(test_size=0.5)

train_dataset = split_ds_1["train"].select(range(100))
eval_dataset = split_ds_2["train"].select(range(100))
test_dataset = split_ds_2["test"].select(range(10))

In [7]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```


# Train

In [8]:
# HF hub ID
BASE_MODEL_NAME = "arnir0/Tiny-LLM"
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [9]:
# Lab (Mandatory from Project Document)
lora_r = 16
lora_target_modules = "all-linear"
bs = 1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

# Others (Recommended / Required for successful run on T4)
lora_alpha = 32                # Standard practice: alpha = 2 * r
lora_dropout = 0.05            # Standard to prevent overfitting
lr_scheduler_type = "cosine"   # Best convergence for LLMs
warmup_ratio = 0.03            # 3% warmup for stability
weight_decay = 0.01


## Wandb

In [10]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

In [11]:
import wandb
wandb.login(key=WANDB_API_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohamed-mohamed-ibrahim (mohamed-mohamed-ibrahim-alexandria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [12]:
config_dict = {
    "base_model": BASE_MODEL_NAME,
    "lora_r": lora_r,
    "lora_target_modules": lora_target_modules,
    "batch_size": bs,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "epochs": epochs,
    "lr": lr,
    "optim": optim,
    "max_length": max_length,

    "lora_alpha": lora_alpha,
    "lora_dropout": lora_dropout,
    "scheduler_type": lr_scheduler_type,
    "warmup_ratio": warmup_ratio,
    "quantization": "4bit-nf4",
    "weight_decay": weight_decay
}

# Dynamic run name using the variable
# Example: Qwen/Qwen2-1.5B-Instruct-lora16-bs1-lr0.0002
run_name = f"{BASE_MODEL_NAME.split('/')[-1]}-lora{lora_r}-bs{bs}-lr{lr}"

wandb.init(
    project="sft-nlp-4",
    config=config_dict,
    name=run_name,
    tags=["SFT", "Part-II", BASE_MODEL_NAME.split('/')[-1]]
)

In [13]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [15]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": torch.cuda.current_device()},
    trust_remote_code=True,
    # use_cache=False,
)

# 2. Critical for Qwen/Llama: Ensures dense layers are optimized for training
# model.config.pretraining_tp = 1

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [16]:
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,  # Llama-style
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

In [18]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=lr,
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="epoch",
    optim=optim,

    weight_decay=weight_decay,

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=max_length,
    packing=False,

    # Scheduler
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,

    # Compute loss on model answers only
    completion_only_loss=True,

    # Logging
    logging_steps=50,
    report_to="wandb", # enables logging to W&B 😎

    bf16=True,
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,                # SFTConfig object
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [19]:
# from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

# # 1. Setup the Collator (Critical for "Completion Only" requirement)
# # This tells the model: "Read the Instruction, but only learn from the Response"
# response_template = "### Response:"
# collator = DataCollatorForCompletionOnlyLM(
#     response_template=response_template,
#     tokenizer=tokenizer
# )

# # 2. Define Configuration
# config = SFTConfig(
#     # --- Standard TrainingArguments parameters ---
#     output_dir=OUTPUT_DIR,
#     learning_rate=lr,
#     num_train_epochs=epochs,
#     per_device_train_batch_size=bs,
#     gradient_accumulation_steps=gradient_accumulation_steps,
#     save_strategy="epoch",



#     optim=optim,

#     # --- SFT-specific parameters ---
#     # Note: max_length is often deprecated in favor of max_seq_length in Config
#     max_seq_length=max_length,
#     packing=False,

#     # Scheduler
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.03, # Reverted to 0.03 (Recommended) from 0.1

#     # Logging
#     logging_steps=25,  # More frequent logging is better for short runs
#     report_to="wandb",

#     # Note: We REMOVED 'completion_only_loss=True' here.
#     # We use the 'collator' below instead. It is much safer.
#     dataset_text_field="text", # Ensure your formatting function creates this column
# )

# # 3. Initialize the SFTTrainer
# trainer = SFTTrainer(
#     model=model,
#     args=config,
#     train_dataset=train_dataset,
#     # eval_dataset=eval_dataset, # Optional: SFT often skips eval to save time
#     peft_config=lora_config,

#     # PASS THE COLLATOR HERE
#     data_collator=collator,
# )

## Loop

In [20]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


TrainOutput(global_step=25, training_loss=1.1936947631835937, metrics={'train_runtime': 116.5712, 'train_samples_per_second': 0.858, 'train_steps_per_second': 0.214, 'total_flos': 110495482235904.0, 'train_loss': 1.1936947631835937, 'entropy': 1.28387694388628, 'num_tokens': 13859.0, 'mean_token_accuracy': 0.7135800781846047, 'epoch': 1.0})

In [21]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

# 6. Save adapter weights

In [22]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

Adapter saved to: ./qlora-peft-output/adapter


In [23]:
!tree


.
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── added_tokens.json
│   │   ├── chat_template.jinja
│   │   ├── merges.txt
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── vocab.json
│   ├── checkpoint-25
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── added_tokens.json
│   │   ├── chat_template.jinja
│   │   ├── merges.txt
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── trainer_state.json
│   │   ├── training_args.bin
│   │   └── vocab.json
│   └── README.md
└── wandb
    ├── debug-internal.log -> run-20251225_085048-lnzljnm1/logs/debug-internal.log
    ├── debug.log -> run-20251225_085048-lnzljnm1/logs/debug.log
    ├── latest-run -> run-20251

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# Load merge model

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "arnir0/Tiny-LLM"
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...
Loading adapter onto base model...
Merging LoRA adapter into base model weights...

Merged model saved to: ./merged-model



In [25]:
!tree

.
├── merged-model
│   ├── added_tokens.json
│   ├── chat_template.jinja
│   ├── config.json
│   ├── generation_config.json
│   ├── merges.txt
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── vocab.json
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── added_tokens.json
│   │   ├── chat_template.jinja
│   │   ├── merges.txt
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── vocab.json
│   ├── checkpoint-25
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── added_tokens.json
│   │   ├── chat_template.jinja
│   │   ├── merges.txt
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── trainer_

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# Evaluation

In [26]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [27]:
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"Output: {p['output']}\n")
    output = generate(wrapped)
    print(output)
    print("\n", "-" * 120, "\n")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


==================== MERGED MODEL OUTPUT ====================

Output: You can use the numpy.argmax() function to get the index of the maximum value in each row of array A. Then, you can use advanced indexing to get the corresponding values in array B.

Here's an example code:

``` python
import numpy as np

# create dummy arrays
A = np.random.rand(5, 1000)
B = np.random.rand(5, 1000)

# get indices of maximum values in each row of A
max\_indices = np.argmax(A, axis=1)

# use advanced indexing to get corresponding values in B
result = B[np.arange(B.shape[0]), max\_indices]

print(result)
```

This should output a 1-dimensional array of length 5, where each element corresponds to the maximum value in the corresponding row of A and its corresponding value in B.

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: I am new to Python and still cannot call myself a Python programmer. Speaking of that, please bear with me if my quest

In [28]:
trainer.evaluate()

{'eval_loss': 1.1959291696548462,
 'eval_runtime': 51.0775,
 'eval_samples_per_second': 1.958,
 'eval_steps_per_second': 0.255,
 'eval_entropy': 1.2332818187200105,
 'eval_num_tokens': 13859.0,
 'eval_mean_token_accuracy': 0.7187461944726797,
 'epoch': 1.0}

In [29]:
metrics = trainer.evaluate()
ppl = math.exp(metrics['eval_loss']) if metrics['eval_loss'] < 20 else float('inf')
print('KL-SFT post-train eval_loss:', metrics['eval_loss'], 'ppl:', ppl)
print('Eval losses over steps:', [(h.get('step'), h.get('eval_loss')) for h in trainer.state.log_history if 'eval_loss' in h])


KL-SFT post-train eval_loss: 1.1959291696548462 ppl: 3.3066287626504636
Eval losses over steps: [(25, 1.1959291696548462), (25, 1.1959291696548462)]


# Resources

- Wandb
  - https://wandb.ai/capecape/alpaca_ft/reports/How-to-Fine-tune-an-LLM-Part-3-The-HuggingFace-Trainer--Vmlldzo1OTEyNjMy
  - https://youtu.be/LQvRhQwDOm0
- LoRA
  - https://youtu.be/t1caDsMzWBk
- SFT
  - https://youtu.be/cayFaWkI39A
  - https://youtu.be/XOlyOWE4YaM
  - https://huggingface.co/docs/trl/en/sft_trainer